In [1]:
from pathlib import Path
import os
import sys

project_root = Path.cwd().resolve()
while not (project_root / "src").is_dir() and project_root.parent != project_root:
    project_root = project_root.parent

if not (project_root / "src").is_dir():
    raise RuntimeError("Could not locate the project root.")

os.chdir(project_root)
sys.path.insert(0, str(project_root))

In [2]:
from src.core.config import Config
from src.document.chunker import filter_sections
from src.document.pdf_parser import parse_pdf_document
from src.document.word_parser import parse_word_document

config = Config()

sections = parse_word_document("data/raw/Report.docx")
print(f"Word: {len(sections)} sections, {sum(s['type'] == 'table' for s in sections)} tables")
for section in sections[:5]:
    print(f"  [{section['type']}] {section['title'][:50]} — {len(section['content'])} chars | kind={section.get('section_kind', 'body')}")

sections = parse_pdf_document("data/raw/HuMengqing.pdf", config)
print(f"PDF: {len(sections)} sections, {sum(s['type'] == 'table' for s in sections)} tables")
for section in sections[:5]:
    print(f"  [{section['type']}] {section['title'][:50]} — page {section['page']} — {len(section['content'])} chars")
filtered_sections = filter_sections(sections, config)
print(f"Retrievable PDF sections after chunk-stage filtering: {len(filtered_sections)}")



Word: 104 sections, 23 tables
  [text] Introduction — 595 chars | kind=front_matter
  [text] AUFGABENSTELLUNG — 173 chars | kind=front_matter
  [text] Selbstständigkeitserklärung — 509 chars | kind=front_matter
  [text] Abstract — 1746 chars | kind=abstract
  [text] TABLE OF CONTENTS — 1763 chars | kind=toc
PDF: 84 sections, 10 tables
  [text] Introduction — page 1 — 65 chars
  [text] Development of in-line monitoring of an additive t — page 1 — 406 chars
  [text] Forschungspraktikum Nr.: 85 — page 1 — 126 chars
  [text] Topic: Development of in-line monitoring of an add — page 1 — 2391 chars
  [text] SELBSTSTÄNDIGKEITSERKLÄRUNG — page 3 — 396 chars
Retrievable PDF sections after chunk-stage filtering: 58


In [3]:
from src.core.config import Config

from src.document.word_parser import parse_word_document
sections = parse_word_document("data/raw/Report.docx")
print(f"Total: {len(sections)} sections")
print()
for s in sections:
    title = s["title"][:60]
    chars = len(s["content"])
    print(f"  {chars:>6} chars | [{s['type']:>5}] {title}")

Total: 104 sections

     595 chars | [ text] Introduction
     173 chars | [ text] AUFGABENSTELLUNG
     509 chars | [ text] Selbstständigkeitserklärung
    1746 chars | [ text] Abstract
    1763 chars | [ text] TABLE OF CONTENTS
    2152 chars | [ text] List of Figures
     173 chars | [ text] List of table
      57 chars | [ text] LIST OF ABBREVIATIONS AND SYMBOLS
     765 chars | [table] LIST OF ABBREVIATIONS AND SYMBOLS
      51 chars | [ text] LIST OF ABBREVIATIONS AND SYMBOLS
     218 chars | [table] LIST OF ABBREVIATIONS AND SYMBOLS
    3714 chars | [ text] 1 Introduction
    2395 chars | [ text] 2.1 Additive Manufacturing
    2735 chars | [ text] 2.2 Optical Coherence Tomography
    1104 chars | [ text] 2.3 Artificial Neural Networks
     435 chars | [ text] 2.3.1 Neuron
     195 chars | [table] 2.3.1 Neuron
     417 chars | [ text] 2.3.1 Neuron
     331 chars | [ text] 2.3.2 Activation Function
     178 chars | [table] 2.3.2 Activation Function
     262 chars | [ text] 2.3.2 

In [4]:
from src.core.config import Config
from src.document.pdf_parser import parse_pdf_document

config = Config()
sections = parse_pdf_document("data/raw/HuMengqing.pdf", config)

print(f"Total: {len(sections)} sections")
print()

for section in sections:
    title = section["title"][:60]
    character_count = len(section["content"])
    page_number = section.get("page")
    page_label = str(page_number) if page_number is not None else "?"

    print(
        f"  page {page_label:>3} | {character_count:>6} chars | "
        f"[{section['type']}] {title} | kind={section.get('section_kind', 'body')}"
    )

Total: 84 sections

  page   1 |     65 chars | [text] Introduction | kind=front_matter
  page   1 |    406 chars | [text] Development of in-line monitoring of an additive thermoplast | kind=front_matter
  page   1 |    126 chars | [text] Forschungspraktikum Nr.: 85 | kind=front_matter
  page   1 |   2391 chars | [text] Topic: Development of in-line monitoring of an additive ther | kind=front_matter
  page   3 |    396 chars | [text] SELBSTSTÄNDIGKEITSERKLÄRUNG | kind=front_matter
  page   4 |   1278 chars | [text] ABSTRACT | kind=abstract
  page   5 |   1619 chars | [text] CONTENTS | kind=toc
  page   7 |   2542 chars | [text] LIST OF FIGURES | kind=toc
  page   9 |    459 chars | [text] LIST OF TABLES | kind=toc
  page  10 |    994 chars | [table] Table 1 | kind=body
  page  10 |    409 chars | [table] Table 2 | kind=body
  page  11 |   3848 chars | [text] 1 Introduction and Motivation | kind=body
  page  13 |    650 chars | [text] 2 Theoretical Basis and Current Situation | kind=bod

In [5]:
import json
from pathlib import Path

from src.core.config import Config
from src.document.pdf_parser import parse_pdf_document
from src.document.word_parser import parse_word_document

output_directory = Path("output/v2/sections")
output_directory.mkdir(parents=True, exist_ok=True)

documents = {
    Path("data/raw/Report.docx"): parse_word_document,
    Path("data/raw/HuMengqing.pdf"): lambda document_path: parse_pdf_document(
        document_path, Config()
    ),
}

for document_path, parser in documents.items():
    sections = parser(document_path)
    output_path = output_directory / f"{document_path.stem}.json"
    output_path.write_text(
        json.dumps(sections, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(f"Saved {len(sections)} sections to {output_path}")


Saved 104 sections to output/v2/sections/Report.json
Saved 84 sections to output/v2/sections/HuMengqing.json


In [6]:
import json
from pathlib import Path

from src.core.config import Config
from src.document.chunker import prepare_chunks
from src.document.pdf_parser import parse_pdf_document
from src.document.word_parser import parse_word_document

output_directory = Path("output/v2/chunks")
filtered_sections_directory = Path("output/v2/filtered_sections")
output_directory.mkdir(parents=True, exist_ok=True)
filtered_sections_directory.mkdir(parents=True, exist_ok=True)
config = Config()

documents = {
    Path("data/raw/Report.docx"): parse_word_document,
    Path("data/raw/HuMengqing.pdf"): lambda document_path: parse_pdf_document(
        document_path, config
    ),
}
all_chunks = []

for document_path, parser in documents.items():
    sections = parser(document_path)
    filtered_sections, chunks = prepare_chunks(sections, config)
    filtered_sections_path = filtered_sections_directory / f"{document_path.stem}.json"
    filtered_sections_path.write_text(
        json.dumps(filtered_sections, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(f"Retained {len(filtered_sections)} sections for retrieval: {filtered_sections_path}")
    all_chunks.extend(chunks)
    output_path = output_directory / f"{document_path.stem}_chunks.json"
    output_path.write_text(
        json.dumps(chunks, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(f"Saved {len(chunks)} chunks to {output_path}")

all_chunks_path = output_directory / "all_chunks.json"
all_chunks_path.write_text(
    json.dumps(all_chunks, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"Saved {len(all_chunks)} chunks to {all_chunks_path}")


Retained 97 sections for retrieval: output/v2/filtered_sections/Report.json
Saved 172 chunks to output/v2/chunks/Report_chunks.json
Retained 58 sections for retrieval: output/v2/filtered_sections/HuMengqing.json
Saved 138 chunks to output/v2/chunks/HuMengqing_chunks.json
Saved 310 chunks to output/v2/chunks/all_chunks.json
